# 06 — Cleanup

Removes the objects created by this demo. The catalog and Vector Search **endpoint**
are preserved by default (they may be shared) — uncomment the last cells to remove
them too.

In [0]:
CATALOG = "bigdata_demo"
SCHEMA = "financial_intelligence"
VS_ENDPOINT = "bigdata_demo_vs"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.research_docs_index"
UC_MODEL_NAME = f"{CATALOG}.{SCHEMA}.financial_intelligence_agent"
ENDPOINT_NAME = "agents_bigdata_demo-financial_intelligence-financial_intelligence_agent"

## 1. Delete the deployed serving endpoint

In [0]:
try:
    from databricks import agents
    agents.delete_deployment(UC_MODEL_NAME)
    print("Deleted agent deployment.")
except Exception as e:
    print(f"Skip deployment delete: {e}")

## 2. Delete the Vector Search index

In [0]:
try:
    from databricks.vector_search.client import VectorSearchClient
    VectorSearchClient(disable_notice=True).delete_index(
        endpoint_name=VS_ENDPOINT, index_name=INDEX_NAME
    )
    print("Deleted vector index.")
except Exception as e:
    print(f"Skip index delete: {e}")

## 3. Drop tables, functions, and the registered model

In [0]:
for fn in ["get_top_holdings", "get_portfolio_positions", "get_ticker_exposure"]:
    spark.sql(f"DROP FUNCTION IF EXISTS {CATALOG}.{SCHEMA}.{fn}")

for tbl in ["accounts", "portfolios", "holdings", "transactions", "research_documents"]:
    spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.{tbl}")

spark.sql(f"DROP FUNCTION IF EXISTS {UC_MODEL_NAME}")  # no-op if not a function
print("Dropped tables and functions.")

## 4. (Optional) Remove the Vector Search endpoint, MCP Service, and catalog

In [0]:
# from databricks.vector_search.client import VectorSearchClient
# VectorSearchClient(disable_notice=True).delete_endpoint(VS_ENDPOINT)

# Governed MCP path only (delete via REST):
# databricks api delete /api/2.1/unity-catalog/mcp-services/bigdata_demo.financial_intelligence.bigdata_mcp
# spark.sql("DROP CONNECTION IF EXISTS bigdata_demo.financial_intelligence.bigdata_http")

# spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{SCHEMA} CASCADE")
# spark.sql(f"DROP CATALOG IF EXISTS {CATALOG} CASCADE")

Cleanup complete. The Databricks secret scope `bigdata` is left in place — remove it
with `databricks secrets delete-scope bigdata` if you no longer need the API key.